# CSP Scanner — Historische Analyse (DuckDB)

Dieses Notebook liest direkt aus der DuckDB-Datenbank.  
Kein IB-Connect notwendig — reine Datenanalyse.

**Voraussetzung:** Mindestens ein Scan-Lauf wurde mit `store.enabled: true` in `settings.yaml` ausgeführt.

## 0 — Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.store import ScanStore

DB_PATH = PROJECT_ROOT / 'data' / 'csp_history.duckdb'

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

store = ScanStore(DB_PATH)
print(f'Connected to: {store.path}')

---
## 1 — Scan-Runs Übersicht

In [ ]:
df_runs = store.runs()
print(f'{len(df_runs)} Runs in der Datenbank:\n')
df_runs

---
## 2 — Letzter Run: Top-Kandidaten

In [ ]:
df_latest = store.query('''
    SELECT
        symbol, expiry_date, dte, strike, spot,
        ROUND(moneyness   * 100, 2) AS moneyness_pct,
        ROUND(ann_yield   * 100, 2) AS ann_yield_pct,
        ROUND(total_yield * 100, 2) AS total_yield_pct,
        ROUND(spread_pct  * 100, 2) AS spread_pct,
        iv, delta, breakeven, score, rating
    FROM latest_run
    ORDER BY score DESC
''').df()

print(f'{len(df_latest)} Kandidaten im letzten Run\n')
df_latest.head(20)

---
## 3 — Rendite-Entwicklung pro Symbol (alle Runs)

In [ ]:
df_trend = store.query('''
    SELECT
        symbol,
        run_ts::DATE                    AS scan_date,
        ROUND(AVG(ann_yield)  * 100, 3) AS avg_yield_pct,
        ROUND(MAX(ann_yield)  * 100, 3) AS max_yield_pct,
        ROUND(AVG(iv)         * 100, 3) AS avg_iv_pct,
        ROUND(AVG(score),     0)        AS avg_score,
        COUNT(*)                        AS n_candidates
    FROM scan_candidates
    GROUP BY symbol, scan_date
    ORDER BY symbol, scan_date
''').df()

print(df_trend.to_string(index=False))

In [ ]:
# Trend-Chart (nur wenn mehrere Runs vorhanden)
symbols = df_trend['symbol'].unique()

if len(df_runs) > 1:
    fig, axes = plt.subplots(len(symbols), 1,
                             figsize=(12, 4 * len(symbols)),
                             squeeze=False)
    for ax, sym in zip(axes[:, 0], symbols):
        grp = df_trend[df_trend['symbol'] == sym]
        ax.plot(grp['scan_date'], grp['avg_yield_pct'], marker='o', label='Ø Ann. Rendite %')
        ax.plot(grp['scan_date'], grp['avg_iv_pct'],    marker='s', linestyle='--', label='Ø IV %')
        ax.set_title(sym, fontsize=13, fontweight='bold')
        ax.set_ylabel('%')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
        ax.legend()
        ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print('Noch nicht genug Runs für einen Trend-Chart (mindestens 2 Runs benötigt).')

---
## 4 — Filtered Query: Attraktive PLTR-Kandidaten (Beispiel)

In [ ]:
# Alle PLTR-Kandidaten mit Yield >= 9% aus allen Runs
df_pltr = store.candidates(symbol='PLTR', min_yield=0.09).df()

cols = ['run_ts', 'expiry_date', 'dte', 'strike', 'spot',
        'ann_yield', 'total_yield', 'breakeven', 'score', 'rating']
print(f'{len(df_pltr)} PLTR-Kandidaten mit Yield >= 9%:\n')
df_pltr[cols].head(20)

---
## 5 — Cross-Run: Beste Strike/Expiry-Kombination

Welche Kombinationen tauchen **konsistent** als attraktiv auf?

In [ ]:
df_consistent = store.query('''
    SELECT
        symbol,
        strike,
        expiry,
        COUNT(DISTINCT run_id)          AS n_runs_seen,
        ROUND(AVG(ann_yield)  * 100, 2) AS avg_yield_pct,
        ROUND(AVG(score), 0)            AS avg_score,
        ROUND(AVG(delta), 4)            AS avg_delta,
        ROUND(AVG(iv)     * 100, 2)     AS avg_iv_pct
    FROM scan_candidates
    GROUP BY symbol, strike, expiry
    HAVING COUNT(DISTINCT run_id) > 1
    ORDER BY n_runs_seen DESC, avg_score DESC
    LIMIT 30
''').df()

if df_consistent.empty:
    print('Noch nicht genug Runs für Cross-Run-Analyse (mindestens 2 Runs mit gleichen Kontrakten).')
else:
    print(f'{len(df_consistent)} wiederkehrende Kombinationen:\n')
    df_consistent

---
## 6 — Risk/Reward-Matrix (letzter Run)

In [ ]:
df_matrix = store.query('''
    SELECT symbol, strike, dte,
        ann_yield, moneyness, score, rating, iv, delta
    FROM latest_run
    WHERE ann_yield IS NOT NULL AND moneyness IS NOT NULL
''').df()

colors = {'Attraktiv': '#27ae60', 'Mittel': '#f39c12', 'Vorsicht': '#e74c3c'}
symbols = df_matrix['symbol'].unique()

fig, ax = plt.subplots(figsize=(12, 7))
for sym in symbols:
    grp = df_matrix[df_matrix['symbol'] == sym]
    for rating, color in colors.items():
        sub = grp[grp['rating'] == rating]
        if sub.empty:
            continue
        sc = ax.scatter(
            sub['moneyness'] * 100,
            sub['ann_yield'] * 100,
            c=color,
            s=sub['score'] * 3,
            alpha=0.75,
            label=f'{sym} — {rating}',
            edgecolors='white', linewidth=0.5,
        )
        for _, row in sub.iterrows():
            ax.annotate(
                f"{sym} K{row['strike']:.0f}",
                (row['moneyness'] * 100, row['ann_yield'] * 100),
                fontsize=7, alpha=0.7, xytext=(4, 4), textcoords='offset points',
            )

ax.set_xlabel('Moneyness (%) — links = tiefer OTM = mehr Puffer', fontsize=11)
ax.set_ylabel('Ann. Rendite (%)', fontsize=11)
ax.set_title('Risk/Reward-Matrix — letzter Scan\nPunktgröße = Score', fontsize=13)
ax.axvline(0, color='grey', linestyle=':', linewidth=0.8)
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

---
## 7 — Freie SQL-Query

Direkte DuckDB-Abfrage — volle SQL-Power auf allen historischen Daten.

In [ ]:
# Beispiel: Top-10 Kandidaten aller Zeit nach Score
df_sql = store.query('''
    SELECT
        run_ts::DATE AS scan_date,
        symbol,
        expiry_date,
        dte,
        strike,
        ROUND(ann_yield * 100, 2)   AS yield_pct,
        ROUND(moneyness * 100, 2)   AS moneyness_pct,
        score,
        rating
    FROM scan_candidates
    ORDER BY score DESC
    LIMIT 10
''').df()

df_sql

In [ ]:
# Disconnect (optional — GC räumt automatisch auf)
store.close()
print('Store geschlossen.')